# Titanic — Data Generation

this notebook loads the Titanic dataset, cleans it, and generates three datasets with MCAR, MAR, and MNAR missingness using mdatagen.

run this notebook once to reproduce the CSV files used by `../analysis_multi.ipynb`. the CSVs are saved in the same `data/` folder as this notebook.

In [1]:
import numpy as np 
import seaborn as sns

from mdatagen.univariate.uMCAR import uMCAR
from mdatagen.univariate.uMAR import uMAR
from mdatagen.univariate.uMNAR import uMNAR

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import plotly.io as pio
pio.renderers.default = "notebook"

In [2]:
# initial data set loading
_df_raw = sns.load_dataset("titanic")

_df_raw.isna().sum()

_df_raw.shape

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

(891, 15)

We can see that we have too many missing values in the `deck` column, so we will drop it.

As for the missing values in `age` and `embark_town`, we drop the affected rows since handling them is not relevant for this analysis.

In [3]:
_df_clean = _df_raw.drop(columns=["deck"])
_df_clean = _df_clean.dropna()

_df_clean.isna().sum()

_df_clean.shape

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

(712, 14)

Now we have a clean dataset to work with. We restrict to numeric columns and inject missing values into multiple columns simultaneously to simulate realistic multivariate missingness patterns.

In [4]:
# exclude the non-numeric columns for the analysis for simplicity
_df_numeric = _df_clean.select_dtypes(include=[np.number])

_df_numeric.isna().sum()

_df_numeric.shape

# set the seed to ensure reproducibility
np.random.seed(42)

survived    0
pclass      0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

(712, 6)

We also define `X` and `y` here once, since they are the same for all three generators. We pass `y` because the library requires it — `survived` is not a feature in `X` and will not receive any missing values.

In [5]:
X = _df_numeric.copy().drop(columns=["survived"])
y = _df_clean["survived"].to_numpy()

### MCAR generation

Here we'll generate a dataset with MCAR missingness across multiple columns simultaneously.

Missingness is completely random and does not depend on any observed variable.

In [6]:
generator = mMCAR(X=X, y=y, missing_rate=20, seed=42)
df_mcar = generator.random()

df_mcar.isna().sum()

df_mcar.shape

NameError: name 'mMCAR' is not defined

### MAR generation

Here we'll generate a dataset with MAR missingness across multiple columns simultaneously.

Missingness is introduced using the correlated method from mdatagen, which selects which columns receive missing values based on how strongly they correlate with each other. Rows with higher values in correlated columns are more likely to become missing.

In [ ]:
generator = mMAR(X=X, y=y, n_xmiss=3)
df_mar = generator.correlated(missing_rate=20)

df_mar.isna().sum()

df_mar.shape

### MNAR generation

Here we'll generate a dataset with MNAR missingness across two columns simultaneously.

Missingness is introduced using the `MBOV_randomness` method from mdatagen, which removes lower-end values from each column. `age` and `fare` are selected as they are continuous numeric columns, where value-based removal is most meaningful.

In [ ]:
_mnar_columns = ["age", "fare"]

generator = mMNAR(X=X, y=y, n_xmiss=2, threshold=0)
df_mnar = generator.MBOV_randomness(missing_rate=20, randomness=0, columns=_mnar_columns)

df_mnar.isna().sum()

df_mnar.shape

Now we save the generated datasets to CSV so the analysis notebook can load them any time.

In [ ]:
df_mcar.to_csv("mcar_multi.csv", index=False)
df_mar.to_csv("mar_multi.csv", index=False)
df_mnar.to_csv("mnar_multi.csv", index=False)